# DDR19 post Geese - Direct Debit rejections strategy (T-SQL + Python port of the SAS script)

**What it does (same as the SAS script):** builds the monthly Direct Debit rejection contact lists and writes them to Excel
(`MIA0`, `MIA1Plus`, `CompLM`, `Commercial`), now with documentation sheets added.

**How to run:** open in VS Code (Python + Jupyter extensions), pick the kernel that has `sqlalchemy`, `pyodbc`, `pandas`, `openpyxl`,
then **Run All**. You only need to touch Section 1 if something unusual is needed - the month is worked out automatically.

**Design:** every SAS `proc sql` pass-through block is T-SQL running on the same server, writing to `#temp` tables.
The SAS `data` step merges become `LEFT JOIN`s, `first.PrimaryAccount` becomes `ROW_NUMBER()`, and the if/else group logic
becomes a `CASE` chain in the same order. Nothing is pulled into Python until the final extracts, so it will also port to Databricks
later with only table-name changes. The same SQL is in `DDR19_pipeline.sql` if you would rather run it in SSMS.

## 1. Configuration (replaces the SAS libname / %let block)

In [ ]:
import datetime as dt
import time
from pathlib import Path

import pandas as pd

SERVER   = r"SBSPSQLVS101\MISPSQL01"
DATABASE = "P_DW"

# THE ONE INPUT: first day of the current month (SAS: %let monthrun='01Oct2026'd;).
# Every date in the process follows from this. None = first day of today's month; or type it, e.g. dt.date(2025, 8, 1)
MONTH_RUN_DATE = None

CUTOFF1 = 632                        # SAS %let cutoff1
CUTOFF2 = 670                        # SAS %let cutoff2
GROUP_NO_MIA = "0MIA"                # zero + MIA (confirmed)

OUTPUT_DIR     = Path(r"H:\Portfolio Management\Credit Management\Collections Strategies 2019\DD Rejections\Deployment\Outputs CM")
SAS_OUTPUT_DIR = Path(r"H:\Portfolio Management\Credit Management\Collections Strategies 2019\DD Rejections\Deployment\Outputs SAS")  # only for the optional comparison in Section 9

# ---- derived values (SAS: &yymm and &yyyymm) ----
RUN_TS    = dt.datetime.now()
MONTH_RUN = MONTH_RUN_DATE or RUN_TS.date().replace(day=1)
YYMM      = MONTH_RUN.strftime("%y%m")                                       # SAS &yymm   e.g. 2610
YYYYMM    = (MONTH_RUN - dt.timedelta(days=1)).strftime("%Y%m")              # SAS &yyyymm e.g. 202609 (previous month)
OUTPUT_FILE = OUTPUT_DIR / f"{YYMM}_DDR.xlsx"

print(f"Run timestamp : {RUN_TS:%Y-%m-%d %H:%M}")
print(f"Month run     : {MONTH_RUN}   (yymm={YYMM}, yyyymm={YYYYMM})")
print(f"Output file   : {OUTPUT_FILE}")

## 2. Connection

In [ ]:
import urllib.parse
import pyodbc
import sqlalchemy as sa

def pick_driver():
    found = [d for d in pyodbc.drivers() if "SQL Server" in d]
    modern = sorted([d for d in found if d.startswith("ODBC Driver")], reverse=True)
    if not (modern or found):
        raise RuntimeError("No SQL Server ODBC driver found on this machine.")
    return (modern or found)[0]

DRIVER = pick_driver()
conn_str = f"DRIVER={{{DRIVER}}};SERVER={SERVER};DATABASE={DATABASE};Trusted_Connection=yes;"
if DRIVER.startswith("ODBC Driver"):
    conn_str += "TrustServerCertificate=yes;"

engine = sa.create_engine("mssql+pyodbc:///?odbc_connect=" + urllib.parse.quote_plus(conn_str))
# one connection for the whole run so the #temp tables survive between steps
conn = engine.connect().execution_options(isolation_level="AUTOCOMMIT")

who = conn.exec_driver_sql("SELECT @@SERVERNAME, DB_NAME(), CONVERT(VARCHAR(19), GETDATE(), 120)").fetchone()
print("Connected:", tuple(who), "| driver:", DRIVER)

## 3. Helpers - run a step, log it, fetch results

In [ ]:
RUN_LOG = []

def sql_header():
    return (
        "SET NOCOUNT ON;\n"
        f"DECLARE @MonthRun DATETIME = '{MONTH_RUN:%Y%m%d}';\n"
        "DECLARE @RunDate DATETIME = GETDATE();\n"
        f"DECLARE @YYYYMM INT = {int(YYYYMM)};\n"
        f"DECLARE @Cutoff1 INT = {int(CUTOFF1)};\n"
        f"DECLARE @Cutoff2 INT = {int(CUTOFF2)};\n"
        "DECLARE @GroupNoMIA VARCHAR(20) = '" + GROUP_NO_MIA.replace("'", "''") + "';\n"
    )

def scalar(sql):
    return conn.exec_driver_sql(sql).scalar()

def fetch(sql):
    res = conn.exec_driver_sql(sql)
    return pd.DataFrame([tuple(r) for r in res.fetchall()], columns=list(res.keys()))

def run_step(step, description, sql, temp_table):
    t0 = time.time()
    conn.exec_driver_sql(sql_header() + sql)
    rows = scalar(f"SELECT COUNT(*) FROM {temp_table}")
    secs = round(time.time() - t0, 1)
    RUN_LOG.append({"Step": step, "Description": description, "Output table": temp_table, "Rows": rows, "Seconds": secs})
    print(f"Step {step:<4} {temp_table:<12} {rows:>9,} rows  ({secs}s)  - {description}")

## 4. Step 1 - DD rejections for the 1st raise  *(SAS lines 19-109)*
Source: `P_ODS.SKIPCORE.DDS99` (rejections) joined to `DDS06` (rejection reasons).
- Current-month rejections (`dc_reqst_o_dt` = Month run date, the first day of the current month); previous month = one month earlier.
- `RTP_Flag` = reason contains "Refer To Pay".
- `RLM_Flag` = account also had a qualifying rejection on the first day of the previous month; `RTP_NoRLM` = RTP and not RLM.
- Accounts currently in the Debt Respite Scheme (`DatDebtRespiteSchemeDetails`, end date >= today, not ceased) are removed - the only test that uses today's date.

In [ ]:
STEP1_SQL = r"""
IF OBJECT_ID('tempdb..#dd') IS NOT NULL DROP TABLE #dd;

WITH CTE_RTPCurrMonth AS
(
SELECT  d99.accnbr_o AS AccountNumber
        , CONVERT(VARCHAR(4),@MonthRun,12) AS 'DDRej'
        , d06.rej_rsn_u AS 'Rejection_Reason'
        , CASE WHEN d06.rej_rsn_u LIKE ('%Refer To Pay%') THEN 1 ELSE 0 END AS 'RTP_Flag'

FROM    P_ODS.SKIPCORE.DDS99 d99

INNER JOIN  P_ODS.SKIPCORE.DDS06 d06
ON  d99.rej_cde_o = d06.rej_cde_u
AND d06.rej_cnt_u=0
AND d06.dc_cease_o_dt IS NULL

WHERE   d99.pool_id_o='SBS00001'
AND     d99._deleted=0
AND     d99.rej_cde_o>0
AND     d99.dc_reqst_o_dt=@MonthRun /*First day of current month*/
),

CTE_RTPPrevMonth AS
(
SELECT  d99.accnbr_o AS AccountNumber
        , 1 AS 'Flag'

FROM    P_ODS.SKIPCORE.DDS99 d99

INNER JOIN  P_ODS.SKIPCORE.DDS06 d06
ON  d99.rej_cde_o = d06.rej_cde_u
AND d06.rej_cnt_u=0
AND d06.dc_cease_o_dt IS NULL

WHERE   d99.pool_id_o='SBS00001'
AND     d99._deleted=0
AND     d99.rej_cde_o>0
AND     d99.dc_reqst_o_dt=DATEADD(MONTH,-1,@MonthRun) /*First day of previous month*/
),

CTE_DRS AS
(
SELECT  m99.primacno_o AS 'PrimaryAccount'
        , m98.accnbr_o AS 'AccountNumber'
        , da.AccountOrigin
        , CONVERT(DATE,drs._timestamp) AS 'DRS_DateRecorded'
        , CONVERT(DATE,drs.startDate) AS 'DRS_StartDate'
        , CONVERT(DATE,drs.endDate) AS 'DRS_EndDate'
        , drs.arrearsAmount AS 'DRS_Arrears'

FROM    P_ODS.SKIPCORE.DatDebtRespiteSchemeDetails drs

INNER JOIN  P_ODS.SKIPCORE.MOR98 m98
ON  drs.myAccount = m98.oid
AND m98.pool_id_o='SBS00001'
INNER JOIN  P_ODS.SKIPCORE.MOR99 m99
ON  m98.myMOR99 = m99.oid
AND m98.pool_id_o = m99.pool_id_o
AND m98.primacno_o = m99.primacno_o
INNER JOIN  P_DW.dbo.dimAccount da
ON  m98.oid = da.BK_Account
AND m98.pool_id_o = da.AccountPool
AND m98.accnbr_o = da.AccountNumber
AND da.AccountType=10

WHERE   CONVERT(DATE,drs.endDate)>=CONVERT(DATE,@RunDate) /*@RunDate = today: the only place the clock is used*/
AND     drs.ceased=0
)

SELECT  a.*
        , CASE WHEN b.Flag=1 THEN 1 ELSE 0 END AS 'RLM_Flag'
        , CASE WHEN a.RTP_Flag=1 AND ISNULL(b.Flag,0)=0 THEN 1 ELSE 0 END AS 'RTP_NoRLM'
INTO    #dd
FROM    CTE_RTPCurrMonth a
LEFT JOIN   CTE_RTPPrevMonth b
ON  a.AccountNumber = b.AccountNumber

WHERE   a.AccountNumber NOT IN (SELECT AccountNumber FROM CTE_DRS) /*Removes accounts currently under the Debt Respite Scheme*/

ORDER BY a.AccountNumber;
"""

run_step("1", 'DD rejections for the 1st raise (current vs previous month, minus Debt Respite Scheme)', STEP1_SQL, "#dd")

## 5. Step 2 - Accounts with a current or future payment holiday  *(SAS lines 112-157)*
Source: `P_ODS.SKIPCORE.ARR96` (arrangement type 33) with `dimAccount`. One row per account (earliest start, ties by `oid`).
These accounts are removed at the end of Step 7.

In [ ]:
STEP2_SQL = r"""
IF OBJECT_ID('tempdb..#ph') IS NOT NULL DROP TABLE #ph;

SELECT  a.AccountNumber
        , a.PH
        , a.PHStartDate
        , a.PHEndDate
INTO    #ph
FROM    (
        SELECT  a96.accnbr_o AS 'AccountNumber'
                , 1 AS 'PH'
                , CONVERT(DATE,a96.dc_start_o_dt) AS 'PHStartDate'
                , CONVERT(DATE,a96.dc_end_o_dt) AS 'PHEndDate'
                , RANK() OVER(PARTITION BY a96.accnbr_o ORDER BY a96.dc_start_o_dt ASC, a96.oid ASC) AS 'PHOrder'

        FROM    P_ODS.SKIPCORE.ARR96 a96

        INNER JOIN  P_DW.dbo.dimAccount da
        ON  a96.accnbr_o = da.AccountNumber
        AND a96.pool_id_o = da.AccountPool
        AND da.CurrentAccountStatus='Live'
        AND da.AccountSubType='Primary'
        AND da.CurrentAccountClassification IN ('Residential','BTL Commercial','Pure Commercial')
        INNER JOIN  P_ODS.SKIPCORE.MOR98 m98
        ON  da.AccountPool = m98.pool_id_o
        AND da.BK_Account = m98.oid

        WHERE   ((da.AccountOrigin NOT IN ('Amber Homeloans Ltd','North Yorkshire Mortgages Ltd') AND a96.dc_cease_o_dt IS NULL)
                OR  da.AccountOrigin IN ('Amber Homeloans Ltd','North Yorkshire Mortgages Ltd'))
        AND     a96.arr_type_o=33
        AND     a96.dc_start_o_dt<=DATEADD(DAY,-1,DATEADD(MONTH,1,@MonthRun)) /*Last day of current month*/
        AND     a96.dc_end_o_dt>=@MonthRun /*First day of current month*/
        )a

WHERE   a.PHOrder=1
ORDER BY a.AccountNumber;
"""

run_step("2", 'Accounts with a current or future payment holiday (PH)', STEP2_SQL, "#ph")

## 6. Step 3 - Month-end data  *(SAS lines 160-234)*
Source: `factMortgageAccountMonthSS` and its dimensions for `yyyymm` (previous month), pool `SBS00001`.
Builds lender, account type/class, balance, redemption/deed/commercial/recent-completion flags, arrears, MIA, red-status letters
(`ADM98`, codes `RED001`, `RED01E`, `RED01F`) and the behavioural score `BS` (`vwBehaviouralScorecard`, 999 when none).

In [ ]:
STEP3_SQL = r"""
IF OBJECT_ID('tempdb..#me') IS NOT NULL DROP TABLE #me;

SELECT  dpa.AccountNumber AS 'PrimaryAccount'
        , da.AccountNumber
        , CASE  WHEN da.AccountOrigin='Amber Homeloans Ltd' THEN 'AHL' WHEN da.AccountOrigin='North Yorkshire Mortgages Ltd' THEN 'NYM' ELSE 'SBS' END AS 'Lender'
        , CASE  WHEN da.CurrentAccountClassification='Residential' THEN 'Resi' WHEN da.CurrentAccountClassification='BTL Commercial' THEN 'BTLC' ELSE 'Comm' END AS 'AccType'
        , CASE  WHEN dsp.SubPopulationCode='BTL' THEN 'BTL' ELSE 'Resi' END AS 'AccClass'
        , das.AccountStatus AS 'StatusMonthEnd'
        , fma.CapitalBalanceLive AS 'CombBal'
        , CASE  WHEN da.AccountStartDate IS NULL THEN '' ELSE CONVERT(DATE,da.AccountStartDate) END AS 'CompDate'
        , CASE  WHEN das.AccountStatus IN ('Redeemed','Redeemed With Balance','Possession') OR fma.CapitalBalanceLive<=0 THEN 1 ELSE 0 END AS 'RedLastM_Flag'
        , CASE  WHEN da.AccountEndDate>=dm.ReportingMonth THEN 1 ELSE 0 END AS 'RedCurrentM_Flag'
        , CASE  WHEN da.AccountEndDate IS NULL THEN '' ELSE CONVERT(DATE,da.AccountEndDate) END AS 'RedDate'
        , CASE  WHEN da.AccountSubType='Primary' THEN 1 ELSE 0 END AS 'PrimAcc_Flag'
        , CASE  WHEN dp.ProductCode IN ('MIN01','HOM10','HM120','HM121','HM122','HM123','HM124','HM125') THEN 1 ELSE 0 END AS 'Deed_Flag'
        , CASE  WHEN da.CurrentAccountClassification='Pure Commercial' THEN 1 ELSE 0 END AS 'Comm_Flag'
        , CASE  WHEN (DATEDIFF(MONTH,da.AccountStartDate,DATEADD(MONTH,0,DATEADD(MONTH,DATEDIFF(MONTH,0,dm.BatchDate),0)))<=2)
                OR (da.AccountStartDate IS NULL AND fma.CapitalBalanceLive>0) THEN 1 ELSE 0 END AS 'RecentComp_Flag'
        , CASE  WHEN DATEDIFF(MONTH,da.AccountStartDate,@MonthRun)<=1 THEN 1 ELSE 0 END AS 'CompLM_Flag'
        , CASE  WHEN da.AccountStartDate IS NULL OR fma.ContractualArrearsAmount<=0 THEN 0 ELSE fma.ContractualArrearsAmount END AS 'Arrears'
        , CASE  WHEN da.AccountStartDate IS NULL OR fma.ContractualArrearsInMonths<=0 THEN 0 ELSE ROUND(fma.ContractualArrearsInMonths,2) END AS 'MIA'
        , CASE  WHEN da.AccountStartDate IS NOT NULL AND fma.ContractualArrearsInMonths>=1 THEN 1 ELSE 0 END AS 'MIA1P_Flag'
        , CASE  WHEN rs.LetterCode IS NULL THEN '' ELSE rs.LetterCode END AS 'RSLetter'
        , CASE  WHEN rs.RSDate IS NULL THEN '1900-01-01' ELSE rs.RSDate END AS 'RSLetterSent'
        , CASE  WHEN rs.RSPeriod IS NULL THEN 0 ELSE 1 END AS 'RedStat_Flag'
        , CASE  WHEN bs.S3Score IS NULL THEN 999 ELSE FLOOR(bs.S3Score) END AS 'BS'
INTO    #me
FROM    P_DW.dbo.factMortgageAccountMonthSS fma

INNER JOIN  P_DW.dbo.dimAccount da
ON  fma.FK_Account = da.PK_Account
INNER JOIN  P_DW.dbo.dimAccount dpa
ON  fma.FK_PrimaryAccount = dpa.PK_Account
INNER JOIN  P_DW.dbo.dimMonth dm
ON  fma.FK_SnapshotDate = dm.PK_Month
INNER JOIN  P_DW.dbo.dimAccountStates das
ON  fma.FK_AccountStates = das.PK_AccountStates
INNER JOIN  P_DW.dbo.dimSubPopulation dsp
ON  fma.FK_SubPopulation = dsp.PK_SubPopulation
INNER JOIN  P_DW.dbo.dimProduct dp
ON  fma.FK_CurrentMortgageProduct = dp.PK_Product
LEFT JOIN   (
            SELECT  a.AccountNumber, a.RSPeriod, a.LetterCode, a.RSDate
            FROM    (
                    SELECT  a98.accnbr_o AS 'AccountNumber', YEAR(a98.dc_creat_o_dt)*100+MONTH(a98.dc_creat_o_dt) AS 'RSPeriod'
                            , a98.lettcode_o AS 'LetterCode', CONVERT(DATE,a98.dc_creat_o_dt) AS 'RSDate', a98.time_o AS 'LetterTime', a98.oid
                            , RANK() OVER(PARTITION BY a98.accnbr_o ORDER BY a98.dc_creat_o_dt DESC, a98.time_o DESC, a98.oid DESC) AS 'RSLatest'
                    FROM    P_ODS.SKIPCORE.ADM98 a98
                    WHERE   a98.pool_id_o='SBS00001'
                    AND     a98.lettcode_o IN ('RED001','RED01E','RED01F')
                    AND     CONVERT(VARCHAR(6),a98.dc_creat_o_dt,112)=CONVERT(VARCHAR,@YYYYMM)
                    )a
            WHERE   a.RSLatest=1) rs
ON  da.AccountNumber = rs.AccountNumber

LEFT JOIN   P_Modelling.COREMODEL.vwBehaviouralScorecard bs
ON  dm.CalendarMonthYear = CONVERT(VARCHAR,bs.Period)
AND da.AccountNumber = bs.PrimaryAccount

WHERE   dm.CalendarMonthYear=CONVERT(VARCHAR,@YYYYMM)
AND     da.AccountPool='SBS00001'
AND     das.AccountStatus IN ('Live','Possession','Redeemed','Redeemed With Balance')

ORDER BY da.AccountNumber;
"""

run_step("3", 'Month-end data, flags, behavioural score, red-status letters', STEP3_SQL, "#me")

## 7. Step 4 - First merge and de-dup  *(SAS lines 237-249)*
Left join of `#dd` to month-end (`#me`) and payment holiday (`#ph`) on `AccountNumber`, then one row per `PrimaryAccount`,
preferring RTP rows, then primary accounts (ties: lowest `AccountNumber`, which is what the SAS stable sort did).

In [ ]:
STEP4_SQL = r"""
IF OBJECT_ID('tempdb..#dd_merged') IS NOT NULL DROP TABLE #dd_merged;

/* SAS: merge dd(in=a) me(in=b) ph(in=c); by AccountNumber; if a;  ==> LEFT JOINs from #dd */
/* SAS: proc sort by PrimaryAccount descending RTP_Flag descending PrimAcc_Flag; then if first.PrimaryAccount;
   SAS sort is stable and the merge output was ordered by AccountNumber, so AccountNumber ASC is the tie-break. */
SELECT  AccountNumber,
       DDRej,
       Rejection_Reason,
       RTP_Flag,
       RLM_Flag,
       RTP_NoRLM,
       PrimaryAccount,
       Lender,
       AccType,
       AccClass,
       StatusMonthEnd,
       CombBal,
       CompDate,
       RedLastM_Flag,
       RedCurrentM_Flag,
       RedDate,
       PrimAcc_Flag,
       Deed_Flag,
       Comm_Flag,
       RecentComp_Flag,
       CompLM_Flag,
       Arrears,
       MIA,
       MIA1P_Flag,
       RSLetter,
       RSLetterSent,
       RedStat_Flag,
       BS,
       PH,
       PHStartDate,
       PHEndDate
INTO    #dd_merged
FROM    (
        SELECT  dd.AccountNumber,
        dd.DDRej,
        dd.Rejection_Reason,
        dd.RTP_Flag,
        dd.RLM_Flag,
        dd.RTP_NoRLM,
        me.PrimaryAccount,
        me.Lender,
        me.AccType,
        me.AccClass,
        me.StatusMonthEnd,
        me.CombBal,
        me.CompDate,
        me.RedLastM_Flag,
        me.RedCurrentM_Flag,
        me.RedDate,
        me.PrimAcc_Flag,
        me.Deed_Flag,
        me.Comm_Flag,
        me.RecentComp_Flag,
        me.CompLM_Flag,
        me.Arrears,
        me.MIA,
        me.MIA1P_Flag,
        me.RSLetter,
        me.RSLetterSent,
        me.RedStat_Flag,
        me.BS,
        ph.PH,
        ph.PHStartDate,
        ph.PHEndDate,
        ROW_NUMBER() OVER (PARTITION BY me.PrimaryAccount
                           ORDER BY dd.RTP_Flag DESC, me.PrimAcc_Flag DESC, dd.AccountNumber ASC) AS rn
        FROM    #dd dd
        LEFT JOIN   #me me ON me.AccountNumber = dd.AccountNumber
        LEFT JOIN   #ph ph ON ph.AccountNumber = dd.AccountNumber
        ) x
WHERE   x.rn = 1;
"""

run_step("4", 'Merge on AccountNumber + de-dup to one row per PrimaryAccount', STEP4_SQL, "#dd_merged")

## 8. Step 5 - Experian data  *(SAS lines 252-315)*
`P_ODS.EXPERIAN.DCMGen10`, period `yyyymm`, lender `SBS`: customer 1 and customer 2 of each primary account.
`CLUhe90pct` = the higher of the two CLU values >= 0.9; `PDLRecent` = either customer has `PDL_A_05` between 1 and 12.

In [ ]:
STEP5_SQL = r"""
IF OBJECT_ID('tempdb..#ah1') IS NOT NULL DROP TABLE #ah1;
IF OBJECT_ID('tempdb..#ah2') IS NOT NULL DROP TABLE #ah2;
IF OBJECT_ID('tempdb..#exp') IS NOT NULL DROP TABLE #exp;

/* Customer 1 of the primary account */
SELECT  ex.*
INTO    #ah1
FROM    (SELECT b.Period
                , b.PrimaryAccount
                , CASE  WHEN b.SP_F3_36<=0 THEN 0 WHEN b.SP_F3_36>=9000 THEN 0 ELSE CONVERT(FLOAT,b.SP_F3_36)/100  END AS 'CLU_AH1'
                , CASE  WHEN b.PDL_A_05 BETWEEN 1 AND 12 THEN 1 ELSE 0 END AS 'PDL_AH1'
        FROM    (
                SELECT  a.*, RANK() OVER(PARTITION BY a.Period, a.LenderCode, a.PrimaryAccount ORDER BY a.CustomerID ASC) AS 'CustRank'
                FROM    P_ODS.EXPERIAN.DCMGen10 a
                WHERE   a.Period=@YYYYMM
                AND     a.LenderCode='SBS'
                )b
        WHERE b.CustRank=1
        )ex
ORDER BY    ex.PrimaryAccount;

/* Customer 2 of the primary account */
SELECT  ex.*
INTO    #ah2
FROM    (SELECT b.Period
                , b.PrimaryAccount
                , CASE  WHEN b.SP_F3_36<=0 THEN 0 WHEN b.SP_F3_36>=9000 THEN 0 ELSE CONVERT(FLOAT,b.SP_F3_36)/100  END AS 'CLU_AH2'
                , CASE  WHEN b.PDL_A_05 BETWEEN 1 AND 12 THEN 1 ELSE 0 END AS 'PDL_AH2'
        FROM    (
                SELECT  a.*, RANK() OVER(PARTITION BY a.Period, a.LenderCode, a.PrimaryAccount ORDER BY a.CustomerID ASC) AS 'CustRank'
                FROM    P_ODS.EXPERIAN.DCMGen10 a
                WHERE   a.Period=@YYYYMM
                AND     a.LenderCode='SBS'
                )b
        WHERE b.CustRank=2
        )ex
ORDER BY    ex.PrimaryAccount;

/* SAS: merge ah1(in=a) ah2(in=b); if a; missing CLU_AH2/PDL_AH2 -> 0;
        MaxCLU=max(of CLU_AH1-CLU_AH2) (SAS max ignores missing);  CLUhe90pct = MaxCLU>=0.9;  PDLRecent = PDL_AH1=1 or PDL_AH2=1 */
SELECT  a.PrimaryAccount
        , CASE WHEN m.MaxCLU>=0.9 THEN 1 ELSE 0 END AS CLUhe90pct
        , CASE WHEN a.PDL_AH1=1 OR ISNULL(b.PDL_AH2,0)=1 THEN 1 ELSE 0 END AS PDLRecent
INTO    #exp
FROM    #ah1 a
LEFT JOIN   #ah2 b ON b.PrimaryAccount = a.PrimaryAccount
CROSS APPLY (SELECT CASE WHEN a.CLU_AH1 IS NULL THEN ISNULL(b.CLU_AH2,0)
                         WHEN a.CLU_AH1 >= ISNULL(b.CLU_AH2,0) THEN a.CLU_AH1
                         ELSE ISNULL(b.CLU_AH2,0) END AS MaxCLU) m;
"""

run_step("5", 'Experian data (customer 1 and 2) -> CLUhe90pct, PDLRecent', STEP5_SQL, "#exp")

## 9. Steps 6-7 - Second merge, Groups, Target  *(SAS lines 318-356)*
Adds Experian fields (missing -> 0), assigns `Group` in the SAS order (Commercial, Red/Rep/Deed, 1+MIA, RecentComp, RSRequested, 0MIA, Unknown),
sets `Target`, `CompLM`, and removes accounts with a payment holiday.

In [ ]:
STEP67_SQL = r"""
IF OBJECT_ID('tempdb..#ddr') IS NOT NULL DROP TABLE #ddr;

/* Step 6 (SAS second merge on PrimaryAccount) = LEFT JOIN #exp, missing -> 0.
   Step 7 (groups + Target): each flag only fires if the earlier ones are 0, exactly as the SAS if/else order.
   SAS treats a missing BS as the lowest number (so BS<=cutoff is TRUE); ISNULL(BS,-1) reproduces that. */
SELECT  d.AccountNumber
        , d.DDRej
        , d.Rejection_Reason
        , d.RTP_Flag
        , d.RLM_Flag
        , d.RTP_NoRLM
        , d.PrimaryAccount
        , d.Lender
        , d.AccType
        , d.AccClass
        , d.StatusMonthEnd
        , d.CombBal
        , d.CompDate
        , d.RedLastM_Flag
        , d.RedCurrentM_Flag
        , d.RedDate
        , d.PrimAcc_Flag
        , d.Deed_Flag
        , d.Comm_Flag
        , d.RecentComp_Flag
        , d.CompLM_Flag
        , d.Arrears
        , d.MIA
        , d.MIA1P_Flag
        , d.RSLetter
        , d.RSLetterSent
        , d.RedStat_Flag
        , d.BS
        , ISNULL(e.CLUhe90pct,0) AS CLUhe90pct
        , ISNULL(e.PDLRecent,0) AS PDLRecent
        , g.[Group] AS [Group]
        , CASE WHEN d.RTP_NoRLM=1 AND g.[Group]=@GroupNoMIA AND ISNULL(d.BS,-1)<=@Cutoff1 THEN 1
             WHEN d.RTP_NoRLM=1 AND g.[Group]=@GroupNoMIA AND ISNULL(e.PDLRecent,0)=1 THEN 1
             WHEN d.RTP_NoRLM=1 AND g.[Group]=@GroupNoMIA AND ISNULL(d.BS,-1)<=@Cutoff2 AND ISNULL(e.CLUhe90pct,0)=1 THEN 1
             ELSE 0 END AS Target
        , CASE WHEN d.RTP_Flag=1 AND d.CompLM_Flag=1 THEN 1 ELSE 0 END AS CompLM
INTO    #ddr
FROM    #dd_merged d
LEFT JOIN   #exp e ON e.PrimaryAccount = d.PrimaryAccount
CROSS APPLY (SELECT CASE WHEN d.Comm_Flag=1 THEN 1 ELSE 0 END AS PC) s1
CROSS APPLY (SELECT CASE WHEN s1.PC=0 AND (d.RedLastM_Flag=1 OR d.RedCurrentM_Flag=1 OR d.Deed_Flag=1) THEN 1 ELSE 0 END AS RD) s2
CROSS APPLY (SELECT CASE WHEN s1.PC=0 AND s2.RD=0 AND d.MIA1P_Flag=1 THEN 1 ELSE 0 END AS ARR) s3
CROSS APPLY (SELECT CASE WHEN s1.PC=0 AND s2.RD=0 AND s3.ARR=0 AND d.RecentComp_Flag=1 THEN 1 ELSE 0 END AS RC) s4
CROSS APPLY (SELECT CASE WHEN s1.PC=0 AND s2.RD=0 AND s3.ARR=0 AND s4.RC=0 AND d.RedStat_Flag=1 THEN 1 ELSE 0 END AS RSR) s5
CROSS APPLY (SELECT CASE WHEN s1.PC=0 AND s2.RD=0 AND s3.ARR=0 AND s4.RC=0 AND s5.RSR=0 THEN 1 ELSE 0 END AS NoMIA) s6
CROSS APPLY (SELECT CASE WHEN s1.PC=1 THEN 'Commercial'
                         WHEN s2.RD=1 THEN 'Red/Rep/Deed'
                         WHEN s3.ARR=1 THEN '1+MIA'
                         WHEN s4.RC=1 THEN 'RecentComp'
                         WHEN s5.RSR=1 THEN 'RSRequested'
                         WHEN s6.NoMIA=1 THEN @GroupNoMIA
                         ELSE 'Unknown' END AS [Group]) g
WHERE   ISNULL(d.PH,0)<>1   /* SAS: if PH=1 then delete */
ORDER BY d.PrimaryAccount;
"""

run_step("6-7", 'Second merge on PrimaryAccount, Groups, Target, CompLM, remove PH accounts', STEP67_SQL, "#ddr")

## 10. Validation counts and checks

In [ ]:
counts = {
    "n_dd":            scalar("SELECT COUNT(*) FROM #dd"),
    "n_dd_dupe_acct":  scalar("SELECT COUNT(*) - COUNT(DISTINCT AccountNumber) FROM #dd"),
    "n_ph":            scalar("SELECT COUNT(*) FROM #ph"),
    "n_me":            scalar("SELECT COUNT(*) FROM #me"),
    "n_unmatched_me":  scalar("SELECT COUNT(*) FROM #dd d LEFT JOIN #me m ON m.AccountNumber = d.AccountNumber WHERE m.AccountNumber IS NULL"),
    "n_merged":        scalar("SELECT COUNT(*) FROM #dd_merged"),
    "n_ph_removed":    scalar("SELECT COUNT(*) FROM #dd_merged WHERE ISNULL(PH,0) = 1"),
    "n_ah1":           scalar("SELECT COUNT(*) FROM #ah1"),
    "n_ah1_dupe":      scalar("SELECT COUNT(*) - COUNT(DISTINCT PrimaryAccount) FROM #ah1"),
    "n_exp":           scalar("SELECT COUNT(*) FROM #exp"),
    "n_final":         scalar("SELECT COUNT(*) FROM #ddr"),
    "n_final_exp_match": scalar("SELECT COUNT(*) FROM #ddr r INNER JOIN #exp e ON e.PrimaryAccount = r.PrimaryAccount"),
}
counts

## 11. Extracts (one query per Excel sheet)
Same filters and columns as SAS Step 8. Column order follows the SAS dataset order.

In [ ]:
EXTRACT_SQL = {
    "MIA0": r"""
SELECT AccountNumber, DDRej, Rejection_Reason, PrimaryAccount, Lender, AccType, AccClass, Arrears, BS, CLUhe90pct, PDLRecent, [Group], Target
FROM #ddr
WHERE Target=1
ORDER BY PrimaryAccount;
""",
    "MIA1Plus": r"""
SELECT AccountNumber, DDRej, Rejection_Reason, PrimaryAccount, Lender, AccType, AccClass, Arrears, MIA, RSLetter, RSLetterSent, BS, CLUhe90pct, PDLRecent, [Group]
FROM #ddr
WHERE [Group]='1+MIA' AND RTP_NoRLM=1
ORDER BY PrimaryAccount;
""",
    "CompLM": r"""
SELECT AccountNumber, DDRej, Rejection_Reason, PrimaryAccount, Lender, AccType, CompDate, Arrears, MIA, [Group]
FROM #ddr
WHERE CompLM=1
ORDER BY PrimaryAccount;
""",
    "Commercial": r"""
SELECT AccountNumber, DDRej, Rejection_Reason, PrimaryAccount, Lender, Arrears, MIA, BS, CLUhe90pct, PDLRecent, [Group]
FROM #ddr
WHERE Comm_Flag=1
ORDER BY PrimaryAccount;
""",
}
EXTRACT_RULES = {'MIA0': 'Target=1', 'MIA1Plus': "Group='1+MIA' AND RTP_NoRLM=1", 'CompLM': 'CompLM=1', 'Commercial': 'Comm_Flag=1'}

df_ddr = fetch("SELECT * FROM #ddr ORDER BY PrimaryAccount")
extracts = {sheet: fetch(sql) for sheet, sql in EXTRACT_SQL.items()}
for s, d in extracts.items():
    print(f"{s:<11} {len(d):>9,} rows x {d.shape[1]} cols")

## 12. Checks

In [ ]:
def build_checks(df, counts, extracts):
    out = []
    def add(name, status, detail):
        out.append({"Check": name, "Result": status, "Detail": detail})

    ok = MONTH_RUN.day == 1 and (MONTH_RUN.year, MONTH_RUN.month) == (RUN_TS.year, RUN_TS.month)
    add("Month run is first day of this calendar month", "PASS" if ok else "WARN",
        f"Month run {MONTH_RUN}; today {RUN_TS:%Y-%m-%d}. "
        + ("OK." if ok else "Not the first day of the current month - fine for a re-run of a past month, but the Debt Respite Scheme exclusion uses today's date."))
    add("Step 1 returned rejections", "PASS" if counts["n_dd"] > 0 else "FAIL", f"{counts['n_dd']:,} rows in #dd")
    add("Step 3 returned month-end rows", "PASS" if counts["n_me"] > 0 else "FAIL", f"{counts['n_me']:,} rows in #me for {YYYYMM}")
    add("Duplicate AccountNumber in Step 1", "PASS" if counts["n_dd_dupe_acct"] == 0 else "WARN",
        f"{counts['n_dd_dupe_acct']:,} duplicate account rows (same account rejected more than once)")
    add("Rejected accounts with no month-end match", "PASS" if counts["n_unmatched_me"] == 0 else "WARN",
        f"{counts['n_unmatched_me']:,} rows. These have no PrimaryAccount/BS; SAS treats missing BS as lowest, reproduced here.")
    add("One row per PrimaryAccount in final", "PASS" if df["PrimaryAccount"].nunique(dropna=False) == len(df) else "FAIL",
        f"{len(df):,} rows, {df['PrimaryAccount'].nunique(dropna=False):,} distinct PrimaryAccount")
    exp_rows = counts["n_merged"] - counts["n_ph_removed"]
    add("Row reconciliation (de-dup minus PH)", "PASS" if counts["n_final"] == exp_rows == len(df) else "FAIL",
        f"{counts['n_merged']:,} after de-dup - {counts['n_ph_removed']:,} PH = {exp_rows:,}; final {counts['n_final']:,}; downloaded {len(df):,}")
    add("Experian customer-1 unique per PrimaryAccount", "PASS" if counts["n_ah1_dupe"] == 0 else "WARN",
        f"{counts['n_ah1_dupe']:,} duplicates among {counts['n_ah1']:,} rows")
    add("Experian match rate (info)", "INFO",
        f"{counts['n_final_exp_match']:,} of {counts['n_final']:,} final accounts have Experian data "
        f"({(counts['n_final_exp_match'] / counts['n_final'] if counts['n_final'] else 0):.1%}); the rest default to 0")
    n_unknown = int((df["Group"] == "Unknown").sum())
    add("Accounts with Group = Unknown", "PASS" if n_unknown == 0 else "WARN", f"{n_unknown:,}")
    bad = df[(df["Target"] == 1) & ~((df["Group"] == GROUP_NO_MIA) & (df["RTP_NoRLM"] == 1))]
    add("Target only on RTP_NoRLM accounts in group 0MIA", "PASS" if bad.empty else "FAIL", f"{len(bad):,} violations")
    expected = {"MIA0": int((df["Target"] == 1).sum()),
                "MIA1Plus": int(((df["Group"] == "1+MIA") & (df["RTP_NoRLM"] == 1)).sum()),
                "CompLM": int((df["CompLM"] == 1).sum()),
                "Commercial": int((df["Comm_Flag"] == 1).sum())}
    for sheet, n in expected.items():
        got = len(extracts[sheet])
        add(f"Sheet {sheet} row count", "PASS" if got == n else "FAIL", f"sheet {got:,} vs expected {n:,}")
    n_null_pa = int(df["PrimaryAccount"].isna().sum())
    add("Null PrimaryAccount in final", "PASS" if n_null_pa == 0 else "WARN", f"{n_null_pa:,}")
    return pd.DataFrame(out)

checks = build_checks(df_ddr, counts, extracts)
display(checks)
if (checks["Result"] == "FAIL").any():
    raise RuntimeError("One or more checks FAILED - see table above. Excel file NOT written.")

## 13. Optional: compare with the SAS output for the same month
Reads `ddrYYMM.sas7bdat` written by the SAS run and compares it to this run, account by account.
Only meaningful if the SAS and this notebook were run on the same day for the same month.

In [ ]:
def reconcile_with_sas(df_new, sas_path):
    import numpy as np
    sas = pd.read_sas(sas_path, format="sas7bdat", encoding="latin-1")
    for c in sas.columns:
        if sas[c].dtype == object:
            sas[c] = sas[c].astype(str).str.strip()
    a, b = sas.copy(), df_new.copy()
    a["PrimaryAccount"] = a["PrimaryAccount"].astype(str).str.strip()
    b["PrimaryAccount"] = b["PrimaryAccount"].astype(str).str.strip()
    rows = [{"Item": "Row count", "SAS": len(a), "New": len(b), "Differences": len(a) - len(b)}]
    only_sas = set(a["PrimaryAccount"]) - set(b["PrimaryAccount"])
    only_new = set(b["PrimaryAccount"]) - set(a["PrimaryAccount"])
    rows.append({"Item": "PrimaryAccount only in one side", "SAS": len(only_sas), "New": len(only_new), "Differences": len(only_sas) + len(only_new)})
    m = a.merge(b, on="PrimaryAccount", suffixes=("_sas", "_new"))
    compare_cols = ["AccountNumber", "DDRej", "Rejection_Reason", "RTP_Flag", "RLM_Flag", "RTP_NoRLM", "Lender", "AccType", "AccClass",
                    "StatusMonthEnd", "CombBal", "RedLastM_Flag", "RedCurrentM_Flag", "PrimAcc_Flag", "Deed_Flag", "Comm_Flag",
                    "RecentComp_Flag", "CompLM_Flag", "Arrears", "MIA", "MIA1P_Flag", "RSLetter", "RedStat_Flag", "BS",
                    "CLUhe90pct", "PDLRecent", "Group", "Target", "CompLM"]
    for c in compare_cols:
        if f"{c}_sas" not in m or f"{c}_new" not in m:
            continue
        x, y = m[f"{c}_sas"], m[f"{c}_new"]
        if pd.api.types.is_numeric_dtype(x) and pd.api.types.is_numeric_dtype(y):
            diff = ~np.isclose(x.astype(float), y.astype(float), equal_nan=True)
        else:
            diff = x.astype(str).str.strip() != y.astype(str).str.strip()
        rows.append({"Item": f"Column {c}", "SAS": len(m), "New": len(m), "Differences": int(diff.sum())})
    return pd.DataFrame(rows), only_sas, only_new

sas_file = SAS_OUTPUT_DIR / f"ddr{YYMM}.sas7bdat"
recon = None
if sas_file.exists():
    recon, only_sas, only_new = reconcile_with_sas(df_ddr, sas_file)
    display(recon[recon["Differences"] != 0] if (recon["Differences"] != 0).any() else recon.head(3))
else:
    print(f"SAS file not found ({sas_file}) - comparison skipped.")

## 14. Write the Excel file (data sheets + documentation sheets)
Data sheets first, in the same order as SAS, then `README`, `Sheet_Index`, `Run_Log`, `Checks`, `Group_Summary`, `Business_Rules`, `Data_Dictionary`
(and `SAS_Comparison` if Section 13 ran). The whole file is rewritten on each run.

In [ ]:
DICT = [('AccountNumber', 'Account that had the DD rejection; after de-dup, the account kept for its PrimaryAccount.', 'P_ODS.SKIPCORE.DDS99 accnbr_o'), ('DDRej', 'Year-month (YYMM) of the Month run date, as text.', 'Derived from Month run'), ('Rejection_Reason', 'Text reason for the DD rejection.', 'P_ODS.SKIPCORE.DDS06 rej_rsn_u'), ('RTP_Flag', "1 if Rejection_Reason contains 'Refer To Pay', else 0.", 'Derived from Rejection_Reason'), ('RLM_Flag', '1 if the account also had a qualifying DD rejection (same filters) on the first day of the previous month, else 0.', 'DDS99 + DDS06, previous month'), ('RTP_NoRLM', '1 if RTP_Flag=1 and RLM_Flag=0.', 'Derived'), ('PrimaryAccount', 'Primary account number the account belongs to. Final output is one row per PrimaryAccount.', 'dimAccount (via FK_PrimaryAccount)'), ('Lender', 'AHL = Amber Homeloans Ltd; NYM = North Yorkshire Mortgages Ltd; otherwise SBS.', 'dimAccount.AccountOrigin'), ('AccType', 'Resi = Residential; BTLC = BTL Commercial; otherwise Comm.', 'dimAccount.CurrentAccountClassification'), ('AccClass', 'BTL if sub-population code is BTL, otherwise Resi.', 'dimSubPopulation.SubPopulationCode'), ('StatusMonthEnd', 'Account status at the month-end snapshot.', 'dimAccountStates.AccountStatus'), ('CombBal', 'Capital balance (live) at month end.', 'factMortgageAccountMonthSS.CapitalBalanceLive'), ('CompDate', "Account start (completion) date. Blank in source shows as 1900-01-01 because SQL Server converts '' to a date.", 'dimAccount.AccountStartDate'), ('RedLastM_Flag', '1 if month-end status is Redeemed / Redeemed With Balance / Possession, or capital balance <= 0.', 'Derived'), ('RedCurrentM_Flag', '1 if the account end date is on/after the reporting month.', 'dimAccount.AccountEndDate, dimMonth.ReportingMonth'), ('RedDate', 'Account end (redemption) date. Blank shows as 1900-01-01 (see CompDate).', 'dimAccount.AccountEndDate'), ('PrimAcc_Flag', '1 if the account sub-type is Primary.', 'dimAccount.AccountSubType'), ('Deed_Flag', '1 if product code is one of MIN01, HOM10, HM120-HM125.', 'dimProduct.ProductCode'), ('Comm_Flag', '1 if classification is Pure Commercial.', 'dimAccount.CurrentAccountClassification'), ('RecentComp_Flag', '1 if completed within 2 months of the snapshot batch date, or no start date and balance > 0.', 'Derived'), ('CompLM_Flag', '1 if completion date is within 1 calendar month of the Month run date.', 'Derived'), ('Arrears', 'Contractual arrears amount; 0 if none, negative, or no start date.', 'factMortgageAccountMonthSS.ContractualArrearsAmount'), ('MIA', 'Contractual arrears in months, rounded to 2 dp; 0 if none, negative, or no start date.', 'factMortgageAccountMonthSS.ContractualArrearsInMonths'), ('MIA1P_Flag', '1 if the account has a start date and arrears are 1 month or more.', 'Derived'), ('RSLetter', 'Latest red-status letter code in the month (RED001 / RED01E / RED01F), blank if none.', 'P_ODS.SKIPCORE.ADM98'), ('RSLetterSent', 'Date that letter was created; 1900-01-01 if none.', 'P_ODS.SKIPCORE.ADM98'), ('RedStat_Flag', '1 if a red-status letter was created in the month.', 'Derived'), ('BS', 'Behavioural score (S3Score rounded down); 999 if no score.', 'P_Modelling.COREMODEL.vwBehaviouralScorecard'), ('CLUhe90pct', '1 if the higher of customer 1 / customer 2 CLU values is >= 0.9 (CLU = SP_F3_36 / 100; 0 if <=0 or >=9000). 0 if no Experian record.', 'P_ODS.EXPERIAN.DCMGen10'), ('PDLRecent', '1 if either customer has PDL_A_05 between 1 and 12. 0 if no Experian record.', 'P_ODS.EXPERIAN.DCMGen10'), ('Group', 'Segment, first match wins: Commercial, Red/Rep/Deed, 1+MIA, RecentComp, RSRequested, 0MIA, Unknown.', 'Derived (Step 7)'), ('Target', '1 if RTP_NoRLM=1, Group=0MIA and (BS<=cutoff1 OR PDLRecent=1 OR (BS<=cutoff2 AND CLUhe90pct=1)).', 'Derived (Step 7)'), ('CompLM', '1 if RTP_Flag=1 and CompLM_Flag=1.', 'Derived (Step 7)')]

RULES = [('Population', "Direct Debit rejections with request date = first day of the 'current' month, pool SBS00001, rejection code > 0, reason active. Accounts under the Debt Respite Scheme are excluded."), ('De-dup', 'One row per PrimaryAccount, preferring RTP rows, then primary accounts; remaining ties go to the lowest AccountNumber.'), ('Payment holiday', 'Accounts with a current or future payment holiday (arrangement type 33) are removed after groups and Target are set.'), ('Group order', '1 Commercial (Comm_Flag=1); 2 Red/Rep/Deed (redeemed last month or this month, or deed product); 3 1+MIA (arrears >= 1 month); 4 RecentComp; 5 RSRequested (red-status letter); 6 0MIA; else Unknown. Each rule only applies if the earlier ones did not.'), ('Target', 'RTP_NoRLM=1 AND Group=0MIA AND (BS<=cutoff1 OR PDLRecent=1 OR (BS<=cutoff2 AND CLUhe90pct=1)). Cut-offs are in the README sheet.'), ('Sheet MIA0', 'Target=1.'), ('Sheet MIA1Plus', 'Group=1+MIA and RTP_NoRLM=1.'), ('Sheet CompLM', 'CompLM=1 (RTP rejection on an account completed within the last month).'), ('Sheet Commercial', 'Comm_Flag=1 (all pure commercial accounts in the de-duped population).'), ('Dates', "One input drives everything: Month run = first day of the current month (SAS monthrun). Current-month rejections = that date; previous-month rejections (RLM) = one month earlier; payment holiday window = that month; month-end data, red-status letters and Experian = the month before. Only the Debt Respite Scheme exclusion (end date >= today) uses today's date."), ('Carried over from SAS', 'A rejected account with no month-end match has no PrimaryAccount or BS. SAS treats missing BS as lowest so BS<=cutoff is TRUE; reproduced here. The Checks sheet counts these.'), ('Carried over from SAS', 'Text sort order (ORDER BY PrimaryAccount) follows SQL Server collation, SAS sorted by byte value. Row membership is not affected.'), ('Changed vs SAS', 'Output is a real .xlsx file (SAS wrote a file called YYMM_DDR with no extension). Documentation sheets are added after the four data sheets.')]

In [ ]:
def build_docs(df, extracts, checks, counts, run_log, recon=None):
    n_by = lambda s: len(extracts[s])
    readme = [
        ("Title", "DDR19 post Geese - Direct Debit rejections strategy"),
        ("Generated", f"{RUN_TS:%Y-%m-%d %H:%M} (notebook run)"),
        ("Server / database", f"{SERVER} / {DATABASE}"),
        ("Month run (SAS monthrun)", f"{MONTH_RUN} - first day of the current month; the one input that drives all dates"),
        ("yymm / yyyymm", f"{YYMM} / {YYYYMM} (month-end data and Experian period = {YYYYMM})"),
        ("Cut-offs", f"cutoff1 = {CUTOFF1}, cutoff2 = {CUTOFF2}"),
        ("Rows in final dataset", f"{len(df):,} (one per PrimaryAccount)"),
        ("Checks", f"{(checks['Result'] == 'PASS').sum()} pass, {(checks['Result'] == 'WARN').sum()} warn, {(checks['Result'] == 'FAIL').sum()} fail"),
        ("Sheets", "MIA0, MIA1Plus, CompLM, Commercial (contact lists); Sheet_Index, Run_Log, Checks, Group_Summary, Business_Rules, Data_Dictionary (documentation)"),
        ("Source tables", "P_ODS.SKIPCORE: DDS99, DDS06, DatDebtRespiteSchemeDetails, MOR98, MOR99, ARR96, ADM98; P_DW.dbo: dimAccount, dimMonth, dimAccountStates, dimSubPopulation, dimProduct, factMortgageAccountMonthSS; P_Modelling.COREMODEL.vwBehaviouralScorecard; P_ODS.EXPERIAN.DCMGen10"),
        ("Method", "T-SQL on SQL Server into #temp tables (Steps 1-7), extracts per sheet (Step 8). Replaces the SAS script DDR19 post Geese."),
        ("Abbreviations", "DD = Direct Debit; RTP = 'Refer To Pay' rejection reason; RLM = also rejected last month; PH = payment holiday; MIA = months in arrears; BS = behavioural score; CLU / PDL = Experian-derived fields (see Data_Dictionary)."),
    ]
    index_rows = []
    for sheet in extracts.keys():
        index_rows.append({"Sheet": sheet, "Rows": n_by(sheet), "Columns": extracts[sheet].shape[1], "Selection rule": EXTRACT_RULES[sheet],
                           "Fields": ", ".join(extracts[sheet].columns)})
    grp = (df.assign(Lender=df["Lender"].fillna("(none)"))
             .pivot_table(index="Group", columns="Lender", values="PrimaryAccount", aggfunc="count", fill_value=0, margins=True, margins_name="Total")
             .reset_index())
    tgt = df.groupby("Group")["Target"].sum().rename("Target=1").reset_index()
    grp = grp.merge(tgt, on="Group", how="left").fillna({"Target=1": int(df["Target"].sum())})
    cols_in = {c: [s for s, d in extracts.items() if c in d.columns] for c in DICT_FIELDS}
    dd = pd.DataFrame([{"Field": f, "Logic": l, "Source": s, "In sheets": ", ".join(cols_in.get(f, [])) or "(not exported)"} for f, l, s in DICT])
    docs = {
        "README": pd.DataFrame(readme, columns=["Item", "Detail"]),
        "Sheet_Index": pd.DataFrame(index_rows),
        "Run_Log": pd.DataFrame(run_log),
        "Checks": checks,
        "Group_Summary": grp,
        "Business_Rules": pd.DataFrame(RULES, columns=["Topic", "Rule"]),
        "Data_Dictionary": dd,
    }
    if recon is not None:
        docs["SAS_Comparison"] = recon
    return docs

DICT_FIELDS = [f for f, _, _ in DICT]

def autoformat(ws):
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="305496")
    ws.freeze_panes = "A2"
    for i, col in enumerate(ws.columns, start=1):
        longest = max((len(str(c.value)) for c in list(col)[:300] if c.value is not None), default=8)
        ws.column_dimensions[get_column_letter(i)].width = min(max(10, longest + 2), 70)
        if longest > 70:
            for c in list(col)[1:]:
                c.alignment = Alignment(wrap_text=True, vertical="top")

def write_workbook(path, extracts, docs):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with pd.ExcelWriter(path, engine="openpyxl", date_format="YYYY-MM-DD", datetime_format="YYYY-MM-DD HH:MM") as xw:
            for sheet, d in extracts.items():
                d.to_excel(xw, sheet_name=sheet, index=False)
            for sheet, d in docs.items():
                d.to_excel(xw, sheet_name=sheet, index=False)
            for ws in xw.book.worksheets:
                autoformat(ws)
    except PermissionError:
        raise RuntimeError(f"Cannot write {path} - is it open in Excel? Close it and re-run this cell.")
    return path

docs = build_docs(df_ddr, extracts, checks, counts, RUN_LOG, recon)
out_path = write_workbook(OUTPUT_FILE, extracts, docs)
print("Written:", out_path)
for s, d in {**extracts, **docs}.items():
    print(f"  {s:<16} {len(d):>9,} rows")

## 15. Done
Tidy up the connection. The `#temp` tables vanish automatically when it closes.

In [ ]:
conn.close()
engine.dispose()
print('Connection closed.')